# Конспект. Модуль 4: От AdaBoost к Gradient Boosting

## 1. Зачем это нужно и как это связано с предыдущими модулями

В Модуле 3 мы вывели математику: псевдо-остаток — это антиградиент функции потерь, а «шаг» градиентного спуска в пространстве функций — это обученное дерево, приближающее этот антиградиент. Но мы ни разу не построили ни одного реального дерева и не сделали ни одной полной итерации алгоритма — вся работа была на уровне одной формулы и одного набора чисел.

В этом модуле мы закрываем два пробела:
1. **Исторический контекст** — откуда вообще взялась идея бустинга (AdaBoost появился на 5+ лет раньше строгой формулировки Gradient Boosting) и как AdaBoost на самом деле является частным случаем общей теории из Модуля 3.
2. **Практика полной итерации** — впервые пройдём **несколько шагов алгоритма от начала до конца** на конкретных числах: построим реальные (пусть и очень простые) деревья, обновим модель, посчитаем, как убывает ошибка.

## 2. AdaBoost: первый работающий алгоритм бустинга

AdaBoost (Adaptive Boosting, 1995, Freund & Schapire) появился раньше строгой теории градиентного бустинга и решает **немного другую** задачу устройства: вместо того чтобы явно считать градиент функции потерь, он **напрямую перевзвешивает объекты**, придавая больший вес тем, на которых текущий ансамбль ошибается.

### 2.1. Алгоритм пошагово

Обозначения: метки классов `y_i ∈ {-1, +1}` (важно — не `{0,1}`, как в LogLoss, а именно `{-1,+1}`, это стандартная конвенция AdaBoost). `h_m(x)` — слабый классификатор на итерации `m`, предсказывающий `-1` или `+1`.

1. **Инициализация весов объектов:** `w_i = 1/N` для всех `i` — на старте все объекты одинаково важны.
2. **Для `m = 1, ..., M`:**
   a. Обучить слабый классификатор `h_m(x)`, минимизирующий **взвешенную** ошибку классификации:

In [ ]:
      err_m = Σ(i: h_m(x_i)≠y_i) w_i   /   Σ(i=1..N) w_i

b. Посчитать вес самого классификатора в итоговом ансамбле:

In [ ]:
      α_m = 0.5 · ln( (1 - err_m) / err_m )

c. Обновить веса объектов:

In [ ]:
      w_i <- w_i · exp( -α_m · y_i · h_m(x_i) )

затем **нормализовать**, чтобы веса снова суммировались в 1.
3. **Финальное предсказание:** взвешенное голосование всех слабых классификаторов:

In [ ]:
   H(x) = sign( Σ(m=1..M) α_m · h_m(x) )

### 2.2. Разбор формулы обновления весов — что она физически делает

Смотрим на показатель степени `-α_m · y_i · h_m(x_i)`:
- Если объект `i` классифицирован **верно**, то `y_i · h_m(x_i) = +1` (совпадение знаков) -> показатель степени `-α_m` (отрицательный, при `α_m>0`) -> множитель `exp(-α_m) < 1` -> **вес уменьшается**.
- Если объект `i` классифицирован **неверно**, `y_i · h_m(x_i) = -1` -> показатель степени `+α_m` -> множитель `exp(+α_m) > 1` -> **вес увеличивается**.

**Интуиция одной фразой:** после каждой итерации объекты, на которых модель ошиблась, становятся «важнее» для следующего слабого классификатора, а те, что уже классифицированы верно — «менее важны». Это и есть механизм последовательной коррекции ошибок, о котором мы говорили в начале Модуля 3, только реализованный не через градиент, а через явное перевзвешивание.

### 2.3. Численный пример: одна полная итерация AdaBoost

Игрушечные данные (в конвенции AdaBoost, метки `{-1,+1}`):

| i | x | y |
|---|---|---|
| 1 | 1 | +1 |
| 2 | 2 | +1 |
| 3 | 3 | -1 |
| 4 | 4 | -1 |
| 5 | 5 | +1 |

Обратите внимание: точка 5 (`x=5, y=+1`) «выбивается» из общего тренда — простой одномерный порог не сможет классифицировать её верно одновременно с остальными. Это специально, чтобы увидеть механизм адаптации.

**Инициализация:** `w = [0.2, 0.2, 0.2, 0.2, 0.2]`.

**Шаг 1a — ищем лучший stump (порог) по взвешенной ошибке.** Перебор порогов `t∈{1.5, 2.5, 3.5, 4.5}`, стратегия «слева +1, справа -1» или наоборот (перебираем и направление тоже). Лучшим оказывается порог `t=2.5` с правилом «`x ≤ 2.5 -> +1`, иначе `-1`»:
- `x=1->+1` (верно), `x=2->+1` (верно), `x=3->-1` (верно), `x=4->-1` (верно), `x=5->-1` (**неверно**, истина `+1`).
- Ошиблись только на объекте 5 -> `err_1 = w_5 / Σw = 0.2 / 1.0 = 0.2`.

**Шаг 1b — считаем `α_1`:**

In [ ]:
α_1 = 0.5 · ln( (1-0.2)/0.2 ) = 0.5 · ln(4) = 0.5 · 1.3863 = 0.6931

**Шаг 1c — обновляем веса.** Для верно классифицированных (`i=1,2,3,4`): множитель `exp(-0.6931) = 0.5`. Для неверно классифицированного (`i=5`): множитель `exp(+0.6931) = 2.0`.

In [ ]:
w = [0.2·0.5, 0.2·0.5, 0.2·0.5, 0.2·0.5, 0.2·2.0] = [0.1, 0.1, 0.1, 0.1, 0.4]

**Нормализация** (сумма сейчас `0.8`, делим каждый на `0.8`):

In [ ]:
w_normalized = [0.125, 0.125, 0.125, 0.125, 0.5]

**Что произошло:** вес «трудного» объекта 5 вырос с `0.2` до `0.5` — теперь он составляет **половину** суммарного веса всей выборки, хотя это всего 1 объект из 5. На следующей итерации слабый классификатор будет **вынужден** обратить на него значительно больше внимания, иначе взвешенная ошибка окажется большой — именно так AdaBoost «фокусируется» на трудных примерах шаг за шагом.

**Финальное предсказание после `M` итераций** — это не простое голосование большинством (как в bagging, Модуль 2), а **взвешенное** голосование, где вес каждого слабого классификатора `α_m` тем больше, чем **точнее** он был на своей итерации (чем меньше `err_m`, тем больше `α_m` — проверьте это на формуле самостоятельно: при `err_m -> 0`, `α_m -> ∞`; при `err_m -> 0.5` — то есть классификатор не лучше монетки — `α_m -> 0`, такой классификатор вообще не учитывается в голосовании).

## 3. Связь AdaBoost и Gradient Boosting

Это важный теоретический факт, который стоит знать хотя бы на уровне понимания (полный вывод — за пределами этого курса, но интуиция важна): **AdaBoost является частным случаем общей схемы «forward stagewise additive modeling» из Модуля 3**, если в качестве функции потерь взять **экспоненциальную**:

In [ ]:
l(y, F) = exp(-y·F(x)),   y ∈ {-1, +1}

Если вывести антиградиент этой функции потерь по тому же рецепту, что мы применяли для MSE и LogLoss в Модуле 3, и подставить в общий алгоритм «посчитать псевдо-остаток -> обучить слабую модель -> обновить аддитивно» — получившийся алгоритм **математически эквивалентен** процедуре перевзвешивания объектов AdaBoost, описанной выше. Этот результат независимо показали Фридман, Хасти и Тибширани в конце 1990-х — он объединяет «изобретённый на интуиции» AdaBoost и строгую теорию градиентного бустинга в **один и тот же общий фреймворк**, просто с разными функциями потерь:

| Алгоритм | Функция потерь `l(y,F)` | Тип задачи |
|---|---|---|
| AdaBoost | Экспоненциальная: `exp(-y·F(x))` | Классификация, `y∈{-1,+1}` |
| Gradient Boosting (регрессия) | MSE: `(y-F)²` | Регрессия |
| Gradient Boosting (классификация) | LogLoss: `-[y·log(p)+(1-y)·log(1-p)]` | Классификация, `y∈{0,1}` |

**Практический вывод из этого исторического экскурса:** современные библиотеки (LightGBM, CatBoost, XGBoost — Модули 6–8) реализуют **именно общую схему Gradient Boosting**, а не AdaBoost-специфичное перевзвешивание — они позволяют подставлять практически любую дифференцируемую функцию потерь (MSE, LogLoss, MAE, Poisson, кастомные для бизнес-задач) именно потому, что общая формулировка через градиент — гибче и не привязана к конкретному виду экспоненциальной перевзвешивающей формулы AdaBoost.

## 4. Полная итерация Gradient Boosting: разбираем на реальных числах

Теперь — центральная часть модуля. Возьмём регрессионный датасет и пройдём **две полные итерации** алгоритма из Модуля 3 (раздел 8), реально обучая деревья-пни (`max_depth=1`) вручную.

### 4.1. Данные

Один признак `x` (условно — час совершения операции) и таргет `y` (условно — размер потенциального ущерба), с тремя выраженными группами:

| i | x | y |
|---|---|---|
| 1 | 1  | 4  |
| 2 | 2  | 6  |
| 3 | 5  | 14 |
| 4 | 6  | 16 |
| 5 | 9  | 29 |
| 6 | 10 | 31 |

Три группы (`{1,2}`, `{5,6}`, `{9,10}`) выбраны специально: одна «пень» (дерево глубины 1, всего одно разбиение -> 2 листа) физически **не может** идеально разделить данные на 3 уровня за один шаг — понадобится минимум 2 итерации, чтобы уловить всю структуру. Это отличная иллюстрация того, зачем вообще нужны множественные итерации бустинга, а не одно дерево побольше.

### 4.2. Итерация 0 — инициализация

In [ ]:
F_0 = mean(y) = (4+6+14+16+29+31)/6 = 100/6 ≈ 16.667

Псевдо-остатки `r⁽⁰⁾_i = y_i - F_0`:

| i | x | y | r⁽⁰⁾ |
|---|---|---|------|
| 1 | 1 | 4 | -12.667 |
| 2 | 2 | 6 | -10.667 |
| 3 | 5 | 14 | -2.667 |
| 4 | 6 | 16 | -0.667 |
| 5 | 9 | 29 | 12.333 |
| 6 | 10 | 31 | 14.333 |

**Train MSE на этом этапе** (среднее квадратов остатков): `MSE₀ ≈ 106.56` — запомним это число, будем следить, как оно убывает.

### 4.3. Итерация 1 — строим первое дерево-пень

Дерево (`max_depth=1`) перебирает пороги между отсортированными значениями `x`: `1.5, 3.5, 5.5, 7.5, 9.5`, считая взвешенный MSE каждого варианта разбиения (точно так же, как в Модуле 1, только критерий теперь — MSE по псевдо-остаткам, а не по исходному `y`).

Пропуская промежуточные вычисления (они делаются буквально по формуле из Модуля 1, раздел 3 — попробуйте повторить самостоятельно для тренировки): **лучший порог — `x ≤ 7.5`**, дающий взвешенный MSE `≈17.67` — заметно лучше остальных кандидатов (для сравнения: `x≤3.5` даёт `38.5`, `x≤5.5` — `31.44`).

In [ ]:
h_1(x) =  -6.667   если x ≤ 7.5   (группы {1,2,5,6}, среднее их остатков)
h_1(x) =  13.333   если x > 7.5   (группа {9,10}, среднее их остатков)

**Обновляем модель** с learning rate `η=0.5` (специально взят крупный шаг для наглядности расчётов — в реальной практике `η` обычно 0.01–0.3, подробно — в Модуле 5):

In [ ]:
F_1(x) = F_0 + 0.5 · h_1(x)

| Группа | F_1 |
|---|---|
| x ∈ {1,2,5,6} | 16.667 + 0.5·(-6.667) = **13.333** |
| x ∈ {9,10} | 16.667 + 0.5·13.333 = **23.333** |

**Новые псевдо-остатки** `r⁽¹⁾ = y - F_1`:

| i | x | y | F_1 | r⁽¹⁾ |
|---|---|---|-----|------|
| 1 | 1 | 4 | 13.333 | -9.333 |
| 2 | 2 | 6 | 13.333 | -7.333 |
| 3 | 5 | 14 | 13.333 | 0.667 |
| 4 | 6 | 16 | 13.333 | 2.667 |
| 5 | 9 | 29 | 23.333 | 5.667 |
| 6 | 10 | 31 | 23.333 | 7.667 |

**Train MSE после итерации 1:** `MSE₁ ≈ 39.89` — упала более чем в 2.5 раза по сравнению с `MSE₀ ≈ 106.56`.

### 4.4. Итерация 2 — второе дерево-пень, уже на новых остатках

Снова перебираем те же 5 кандидатов порогов, но теперь целевая переменная для дерева — уже `r⁽¹⁾`, а не `r⁽⁰⁾`. Лучший порог на этот раз — **`x ≤ 3.5`** (взвешенный MSE `≈5.17`, заметно лучше альтернатив: `x≤7.5` даёт `17.67`, `x≤1.5` — `22.47`):

In [ ]:
h_2(x) =  -8.333   если x ≤ 3.5   (группа {1,2})
h_2(x) =   4.167   если x > 3.5   (группы {5,6,9,10})

Обратите внимание: второе дерево выбрало **другой** признак-порог, чем первое (`3.5` вместо `7.5`) — оно нашло **оставшуюся** структуру в данных, которую первое дерево не могло уловить, имея только 2 листа. Это наглядная демонстрация того, как последовательность неглубоких деревьев вместе покрывает более сложную зависимость, чем каждое дерево по отдельности.

**Обновляем модель:**

In [ ]:
F_2(x) = F_1(x) + 0.5 · h_2(x)

| Группа | F_1 | F_2 |
|---|---|---|
| x ∈ {1,2} | 13.333 | 13.333 + 0.5·(-8.333) = **9.167** |
| x ∈ {5,6} | 13.333 | 13.333 + 0.5·4.167 = **15.417** |
| x ∈ {9,10} | 23.333 | 23.333 + 0.5·4.167 = **25.417** |

**Финальные псевдо-остатки** `r⁽²⁾ = y - F_2`:

| i | x | y | F_2 | r⁽²⁾ |
|---|---|---|-----|------|
| 1 | 1 | 4 | 9.167 | -5.167 |
| 2 | 2 | 6 | 9.167 | -3.167 |
| 3 | 5 | 14 | 15.417 | -1.417 |
| 4 | 6 | 16 | 15.417 | 0.583 |
| 5 | 9 | 29 | 25.417 | 3.583 |
| 6 | 10 | 31 | 25.417 | 5.583 |

**Train MSE после итерации 2:** `MSE₂ ≈ 13.85`.

### 4.5. Сводка сходимости

| Этап | Train MSE | Во сколько раз ниже, чем на старте |
|---|---|---|
| `F_0` (только константа) | 106.56 | 1× |
| `F_1` (1 дерево) | 39.89 | 2.7× |
| `F_2` (2 дерева) | 13.85 | 7.7× |

Это именно то, что мы теоретически ожидали из общей схемы Модуля 3: каждая итерация **последовательно** уменьшает ошибку, приближая аддитивную модель `F_m = F_0 + η·h_1 + η·h_2 + ...` к истинным значениям `y`. Если продолжить итерации дальше, `MSE` продолжит убывать (в какой-то момент — рискуя переобучиться, если не остановиться вовремя — это будет темой Модуля 5).

## 5. Почему в бустинге используют слабые (неглубокие) деревья, а не сильные

Это критически важный концептуальный вопрос, прямо противоположный тому, что мы выучили про Random Forest в Модуле 2 — стоит сравнить их осознанно.

### 5.1. Что было бы, если использовать глубокое дерево в бустинге

Представим, что на итерации 1 мы бы использовали не пень (`max_depth=1`), а **глубокое** дерево без ограничений — точно такое, как в bagging. Такое дерево способно **идеально** подстроиться под псевдо-остатки `r⁽⁰⁾` (вплоть до нуля ошибки на train, аналогично Модулю 1). Тогда:

In [ ]:
F_1(x) = F_0 + η · h_1(x) ≈ F_0 + η · r⁽⁰⁾ = F_0 + η·(y - F_0)

При `η=1` это буквально даёт `F_1(x) = y` — **идеальное** предсказание уже после первой итерации на **обучающих** данных. Звучит здорово, но это чистое переобучение: дерево не «поняло» закономерность, а буквально **запомнило** каждый остаток индивидуально (в пределе — один лист на объект, как в Модуле 1 без ограничений). На новых данных такая модель будет работать плохо, а главное — **дальнейшие итерации бустинга становятся бессмысленны**: после первой же итерации остатки на train близки к нулю, второму дереву буквально нечего исправлять.

### 5.2. Формальное понятие «слабого learner'а»

В теории бустинга (PAC-learning, откуда изначально пришёл AdaBoost) «слабый классификатор» формально определяется как модель, которая **чуть лучше случайного угадывания** — не обязана быть точной, обязана быть лишь **немного информативной**. Смысл всей конструкции бустинга — то, что **множество слегка информативных** моделей, правильно скомбинированных последовательно, дают **в сумме** сильную модель, тогда как одна «слишком умная» модель на первом же шаге лишает смысла саму последовательную схему.

### 5.3. Bias-Variance взгляд — прямое сравнение с Random Forest

| | Random Forest (Модуль 2) | Gradient Boosting |
|---|---|---|
| Механизм борьбы с ошибкой | Усреднение независимых моделей -> снижает **variance** | Последовательная коррекция остатков -> снижает **bias** |
| Какой должна быть базовая модель | Низкий bias, высокая variance (**глубокое** дерево) | Высокий bias, низкая variance (**неглубокое** дерево/«пень») |
| Почему именно такая базовая модель | Усреднение само гасит variance — не нужно, чтобы каждое дерево было «осторожным» | Ансамбль сам снижает bias за счёт множества шагов — не нужно, чтобы одно дерево делало всю работу; наоборот, это разрушило бы последовательный процесс (см. 5.1) |
| Типичная глубина дерева на практике | Не ограничена (`max_depth=None`) | Небольшая, обычно 3–8 уровней (в LightGBM — регулируется через `num_leaves`, Модуль 7) |

**Ключевая фраза, которую стоит запомнить дословно для собеседования:** *«Bagging снижает variance, поэтому ему нужны низко-смещённые, но высоко-дисперсные базовые модели — глубокие деревья. Boosting снижает bias, поэтому ему нужны высоко-смещённые, но низко-дисперсные базовые модели — неглубокие деревья, оставляющие пространство для последующих итераций»*.

Отметим также, что классический AdaBoost исторически использовал буквально **пни** (`max_depth=1`, «decision stumps») — предельный случай слабого learner'а. Современный градиентный бустинг (Модули 6–8) обычно берёт деревья чуть глубже (3–8 уровней), потому что это даёт лучший баланс между скоростью сходимости (меньше итераций нужно) и риском переобучения на каждом шаге — но принцип «дерево должно оставаться существенно слабее, чем задача требует от финальной модели» остаётся тем же.

## 6. Практика: код

### 6.1. Ручной пример — сверка с sklearn

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor

x = np.array([1, 2, 5, 6, 9, 10]).reshape(-1, 1)
y = np.array([4, 6, 14, 16, 29, 31], dtype=float)

eta = 0.5

# Итерация 0
F0 = y.mean()
F = np.full_like(y, F0)
print(f"F0 = {F0:.3f}, train MSE = {np.mean((y-F)**2):.3f}")

# Итерации 1 и 2
for m in range(1, 3):
    residuals = y - F
    stump = DecisionTreeRegressor(max_depth=1)
    stump.fit(x, residuals)
    h = stump.predict(x)
    F = F + eta * h
    mse = np.mean((y - F) ** 2)
    print(f"Итерация {m}: train MSE = {mse:.3f}")
    print(f"  Предсказания дерева h_{m}: {np.round(h, 3)}")
    print(f"  F_{m}: {np.round(F, 3)}")

Запустив этот код, сверьте `train MSE` на каждом шаге с таблицей из раздела 4.5 — они должны совпасть (с точностью до округления), а значения `F_1`/`F_2` — с таблицами разделов 4.3/4.4.

### 6.2. Сверка с `GradientBoostingRegressor`

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(
    n_estimators=2,
    learning_rate=0.5,
    max_depth=1,
    random_state=42
)
gbr.fit(x, y)

print("Предсказания sklearn GBR:", gbr.predict(x))
print("Наши ручные F_2:", F)  # F после цикла из 6.1

Значения должны практически совпасть — небольшие расхождения возможны из-за деталей реализации (например, sklearn по умолчанию использует чуть другую инициализацию `F_0` или порядок обработки одинаковых порогов), но общая картина и порядок величин будут идентичны ручному расчёту.

### 6.3. AdaBoost на sklearn (для практического знакомства с API)

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

X_ada = np.array([1, 2, 3, 4, 5]).reshape(-1, 1)
y_ada = np.array([1, 1, -1, -1, 1])  # конвенция AdaBoost: {-1, +1}

ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=5,
    learning_rate=1.0,
    random_state=42
)
ada.fit(X_ada, y_ada)
print("Предсказания:", ada.predict(X_ada))
print("Веса alpha каждого слабого классификатора:", ada.estimator_weights_)

Обратите внимание на `estimator_weights_` — это именно те `α_m`, которые мы считали вручную в разделе 2.3; можно сравнить порядок величины с нашим `α_1 ≈ 0.693`.

## 7. Частые вопросы на собеседовании

| Вопрос | На что обратить внимание в ответе |
|---|---|
| Чем AdaBoost отличается от Gradient Boosting концептуально? | AdaBoost явно перевзвешивает объекты по формуле с экспонентой; GB работает через градиент произвольной функции потерь. Формально AdaBoost — частный случай GB с экспоненциальной функцией потерь |
| Почему в бустинге используют неглубокие деревья? | Boosting снижает bias последовательно; слишком «сильное» дерево на одной итерации переобучится на остатках, лишив смысла последующие итерации (residuals станут ~0 на train) |
| Почему предсказания в AdaBoost взвешенные (`α_m`), а не простое голосование? | Модели с меньшей взвешенной ошибкой на своей итерации получают больший вес в финальном ансамбле — это делает голосование чувствительным к качеству каждого отдельного слабого классификатора |
| Что происходит с весами объектов, на которых модель уже уверенно права, при переходе AdaBoost к следующей итерации? | Их вес экспоненциально уменьшается (`exp(-α_m) < 1`) — следующий слабый классификатор будет уделять им меньше внимания в пользу более трудных объектов |
| Может ли `α_m` быть отрицательным в AdaBoost? | Да, если `err_m > 0.5` (классификатор хуже случайного угадывания) — тогда `ln((1-err)/err) < 0`; на практике такое почти не встречается, так как слабый классификатор обычно выбирается как минимизирующий взвешенную ошибку, что даёт `err_m < 0.5` по построению |

## 8. Чек-поинт — попробуйте ответить без подсказок

1. Распишите по шагам, что происходит на итерации `m` алгоритма Gradient Boosting.
2. Почему в бустинге используют слабые (неглубокие) деревья, а не сильные?
3. Что произойдёт с обучением бустинга, если на первой же итерации дерево идеально подгонится под псевдо-остатки (train MSE упадёт почти до нуля)?
4. В чём математическая связь между AdaBoost и общей схемой Gradient Boosting из Модуля 3?
5. Почему второе дерево в разделе 4.4 выбрало другой порог разбиения (`x≤3.5`), чем первое (`x≤7.5`)?

## Ответы для самопроверки

<details>
<summary>Раскрыть после того, как попробуете ответить сами</summary>

1. На итерации `m`: (а) считаем псевдо-остатки `r_i = -∂l(y_i,F)/∂F` при текущем `F=F_{m-1}(x_i)` для каждого обучающего объекта; (б) обучаем **регрессионное** дерево `h_m(x)`, приближающее зависимость `x -> r`; (в) обновляем модель `F_m(x) = F_{m-1}(x) + η·h_m(x)`, добавляя вклад нового дерева с учётом learning rate.

2. Потому что boosting атакует **bias** через последовательную коррекцию: если бы каждое отдельное дерево уже было «сильным» (низкий bias/высокая variance, как в Random Forest), оно идеально подстроилось бы под остатки на первой же итерации, оставив последующим деревьям почти нечего исправлять — это ведёт к переобучению на train и не позволяет ансамблю выстраивать постепенное, устойчивое улучшение. Слабые деревья гарантируют, что каждый шаг вносит небольшой, контролируемый вклад.

3. Train-ошибка резко упадёт почти до нуля уже после первой итерации — модель фактически запомнит обучающую выборку через одно дерево, а не выучит обобщающую закономерность. Дальнейшие итерации не смогут добавить полезной информации (остатки уже ~0), а на новых (тестовых) данных качество будет плохим — классический случай переобучения, аналогичный тому, что было с неограниченным одиночным деревом в Модуле 1.

4. Можно показать (результат Фридман/Хасти/Тибширани), что процедура forward stagewise additive modeling из Модуля 3 — если взять в качестве функции потерь экспоненциальную `l(y,F)=exp(-y·F(x))`, `y∈{-1,+1}` — математически сводится к той же самой процедуре перевзвешивания объектов, которую независимо предложили Фройнд и Шапире в AdaBoost. Иными словами, AdaBoost — частный случай общей схемы Gradient Boosting с конкретным выбором функции потерь, просто исторически изобретённый раньше и другим путём (через эвристику перевзвешивания, а не через явный градиентный спуск).

5. Первое дерево уже «забрало» основную часть структуры данных, разбив выборку на {1,2,5,6} против {9,10} — после этого остатки внутри группы {1,2,5,6} всё ещё содержат **свою** внутреннюю структуру (подгруппы {1,2} и {5,6} имеют разные остатки — см. таблицу раздела 4.3), которую первое дерево, имея только 2 листа, физически не могло уловить одновременно с основным разделением. Второе дерево, обучаясь уже на **новых** остатках после первого шага, находит именно эту оставшуюся закономерность — новый оптимальный порог для **этих** остатков оказывается в другом месте (`x≤3.5`), потому что сама целевая переменная для второго дерева (`r⁽¹⁾`) отличается от целевой переменной первого (`r⁽⁰⁾`).

</details>